In [ ]:
!pip install fastapi uvicorn pydantic nest-asyncio pyngrok diffusers transformers accelerate torch torchvision pillow opencv-python timm psutil
# Install BiRefNet directly from its official repository or via transformers if supported
!pip install timm

In [ ]:
import os
import time
import socket
import asyncio
import cv2
import numpy as np
import torch
from PIL import Image
import nest_asyncio
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import StreamingResponse
import uvicorn
from pyngrok import ngrok
from io import BytesIO
from torchvision import transforms
from transformers import AutoModelForImageSegmentation
from fastapi.middleware.cors import CORSMiddleware
import psutil

# -------------------------------------------------------------
# 🧹 ROBUST PORT CLEANUP (Guaranteed to work on Kaggle Linux)
# -------------------------------------------------------------
def kill_port_owner(port=8000):
    print(f"🔍 Scanning for processes occupying port {port}...")
    killed = False
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            connections = proc.connections(kind='inet')
            for conn in connections:
                if conn.laddr.port == port:
                    print(f"💀 Killing process PID {proc.pid} ({proc.name()}) occupying port {port}...")
                    proc.kill()
                    killed = True
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
            pass
    if not killed:
        print(f"👍 Port {port} is clean and ready.")

kill_port_owner(8000)

# Allow FastAPI to run inside an asynchronous Jupyter notebook environment
nest_asyncio.apply()

app = FastAPI(title="AI Image Production Pipeline API")

# Enable CORS middleware to allow requests from frontend (localhost)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# -------------------------------------------------------------
# 🤖 MODEL INITIALIZATION (Loaded globally onto GPU)
# -------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Utilizing compute device: {device}")

# 1. Load Background Removal Model securely via Hugging Face
print("📥 Loading BiRefNet (Segmentation) from Hugging Face...")
birefnet_model = AutoModelForImageSegmentation.from_pretrained(
    "ZhengPeng7/BiRefNet", 
    trust_remote_code=True
).to(device)
birefnet_model.eval()

# 2. Load Lighting/Composition Model (Using SDXL Inpainting)
from diffusers import StableDiffusionXLInpaintPipeline
print("📥 Loading SDXL Inpainting / Composition pipeline...")
composition_pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1", 
    torch_dtype=torch.float16, 
    variant="fp16"
).to(device)

# -------------------------------------------------------------
# 🛠️ HELPER FUNCTIONS
# -------------------------------------------------------------
def remove_background(input_image: Image.Image):
    """ Extracts alpha mask and separates foreground from background """
    img_size = input_image.size
    
    transform_image = transforms.Compose([
        transforms.Resize((1024, 1024)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    input_tensor = transform_image(input_image).unsqueeze(0).to(device)
    if device == "cuda":
        input_tensor = input_tensor.half()
    
    with torch.no_grad():
        outputs = birefnet_model(input_tensor)
        if isinstance(outputs, (tuple, list)):
            pred = outputs[0][-1]
        else:
            pred = outputs
            
        pred = pred.sigmoid().cpu().squeeze()
        
    mask_img = transforms.ToPILImage()(pred).resize(img_size)
    return mask_img

# -------------------------------------------------------------
# 🌐 API ENDPOINTS
# -------------------------------------------------------------

@app.post("/api/remove-bg")
async def api_remove_bg(
    file: UploadFile = File(...)
):
    """ Endpoint for background removal only """
    print("✂️ Received background removal request")
    contents = await file.read()
    init_img = Image.open(BytesIO(contents)).convert("RGB")
    
    # Run BiRefNet segmentation
    alpha_mask = remove_background(init_img)
    
    # Create RGBA image with transparent background
    rgba_img = init_img.copy()
    rgba_img.putalpha(alpha_mask)
    
    img_io = BytesIO()
    rgba_img.save(img_io, 'PNG')
    img_io.seek(0)
    
    headers = {
        "X-Background-Removal-Source": "Local BiRefNet"
    }
    return StreamingResponse(img_io, media_type="image/png", headers=headers)

@app.post("/api/generate-background")
async def generate_background(
    prompt: str = Form(...)
):
    """ Generates a high quality background image from prompt """
    print(f"🎨 Generating background for prompt: {prompt}")
    blank_img = Image.new("RGB", (1024, 1024), "black")
    full_mask = Image.new("L", (1024, 1024), 255)
    
    with torch.autocast(device):
        generated_bg = composition_pipe(
            prompt=f"{prompt}, high resolution, detailed, photorealistic",
            image=blank_img,
            mask_image=full_mask,
            strength=1.0
        ).images[0]
        
    img_io = BytesIO()
    generated_bg.save(img_io, 'JPEG', quality=90)
    img_io.seek(0)
    return StreamingResponse(img_io, media_type="image/jpeg")

@app.post("/api/process-pipeline")
async def process_image_pipeline(
    image: UploadFile = File(...),
    mask_image: UploadFile = File(None),
    bg_image: UploadFile = File(None),
    lighting_prompt: str = Form(...),
    strength: float = Form(0.35)
):
    """ Relighting & harmonizing composition pipeline """
    print(f"⚡ Relighting request: prompt='{lighting_prompt}', strength={strength}")
    contents = await image.read()
    init_img = Image.open(BytesIO(contents)).convert("RGB")
    
    if mask_image is not None:
        mask_contents = await mask_image.read()
        alpha_mask = Image.open(BytesIO(mask_contents)).convert("L")
    else:
        alpha_mask = remove_background(init_img)
        
    if bg_image is not None:
        bg_contents = await bg_image.read()
        uploaded_bg = Image.open(BytesIO(bg_contents)).convert("RGB").resize(init_img.size)
        
        fg_arr = np.array(init_img)
        bg_arr = np.array(uploaded_bg)
        mask_arr = np.array(alpha_mask)[:, :, None] / 255.0
        
        composite_arr = (fg_arr * mask_arr) + (bg_arr * (1 - mask_arr))
        composite_img = Image.fromarray(composite_arr.astype(np.uint8))
    else:
        composite_img = init_img
        
    orig_size = composite_img.size
    comp_resized = composite_img.resize((1024, 1024))
    mask_resized = alpha_mask.resize((1024, 1024))
    
    prompt = f"Subject with {lighting_prompt}, high resolution, matching lighting and shadows, seamless integration"
    
    with torch.autocast(device):
        output_image = composition_pipe(
            prompt=prompt,
            image=comp_resized,
            mask_image=mask_resized,
            strength=strength
        ).images[0]
        
    output_image = output_image.resize(orig_size)
    
    img_io = BytesIO()
    output_image.save(img_io, 'JPEG', quality=90)
    img_io.seek(0)
    
    return StreamingResponse(img_io, media_type="image/jpeg")

# -------------------------------------------------------------
# 🔗 TUNNEL SETUP & EXECUTOR
# -------------------------------------------------------------
try:
    ngrok.kill()
except:
    pass

NGROK_AUTH_TOKEN = "3FcQRLIJKJ3YchA9RyoTnkhCUpA_55zoa75sKc6pxsBPskTom"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000)
print(f"\n🌍 PUBLIC SERVER ACCESS URL: {public_url}\n")
print("Copy this URL directly into your Web App settings.")

def run_api():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info", loop="asyncio")
    server = uvicorn.Server(config)
    server.run()

# Spin up uvicorn server in a background thread
import threading
threading.Thread(target=run_api, daemon=True).start()

# -------------------------------------------------------------
# 🔍 BOOTSTRAP VERIFICATION (Check if server is listening)
# -------------------------------------------------------------
time.sleep(2.5)
def check_server_boot(port=8000):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(('127.0.0.1', port)) == 0:
            print(f"\n✅ SUCCESS: FastAPI Server is listening on port {port}!")
            print(f"Tunnel exposes API successfully: {public_url}/docs")
        else:
            print(f"\n❌ ERROR: FastAPI Server failed to start on port {port}.")
            print("Please re-run this cell or check for console tracebacks.")

check_server_boot(8000)